# Model Deployment Basics — Colab Notebook

This notebook demonstrates:

- Training a small scikit-learn model (Iris dataset)
- Serializing the model with `joblib`
- Building a minimal Flask REST API that loads the saved model and serves predictions
- Exposing the Flask app in Colab using `pyngrok`


In [ ]:
# Install necessary packages (run this cell first)
!pip install -q scikit-learn joblib flask pyngrok requests

## Train a RandomForest classifier and save it to disk

This cell:
1. Loads the Iris dataset.
2. Splits data into train/test sets.
3. Trains a `RandomForestClassifier`.
4. Prints test accuracy.
5. Creates a `model/` folder and saves a joblib file `iris_rf.joblib` containing the model and metadata (`target_names`).

The saved joblib bundle is what the Flask app will load to serve predictions.


In [ ]:
# Train a simple model and save it to disk
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib
import os

In [ ]:
# Prepare data
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Train model
clf = RandomForestClassifier(n_estimators=50, random_state=42)
clf.fit(X_train, y_train)

RandomForestClassifier(n_estimators=50, random_state=42)

In [ ]:
# Evaluate
acc = clf.score(X_test, y_test)
print(f"Test accuracy: {acc:.4f}")

Test accuracy: 1.0000


In [ ]:
# Save model
os.makedirs('model', exist_ok=True)
model_path = 'model/iris_rf.joblib'
joblib.dump({'model': clf, 'target_names': iris.target_names.tolist()}, model_path)
print('Saved model to', model_path)

Saved model to model/iris_rf.joblib


### So what did joblib actually “do” to the Iris data?

➡ Nothing to the raw Iris dataset.

➡ It did not save the raw data.

➡ It saved the trained model that learned from Iris data.

### So joblib did NOT store:

X_train

y_train

X_test

the dataset

### Only this:

* the trained RandomForestClassifier with its internal parameters

* the class names





So your APIs are light — they load the model instantly and predict.

## *Joblib takes your trained Iris model, compresses it, and saves it so your Flask API can load it instantly without retraining.*

## Create `app.py` — a minimal Flask REST API

This cell writes a small Flask application file `app.py`. Key points:
- It loads the serialized model from `model/iris_rf.joblib`.
- It exposes `/` (simple info) and `/predict` endpoints.
- `/predict` expects a JSON body like: `{"instances": [[...], [...]]}` and returns predictions and readable labels (if available).
- No authentication or input validation beyond basic checks — this is a minimal demo.


In [ ]:
# Create a Flask app.py file that loads the model and serves predictions
app_code = '''
from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)

model_bundle = joblib.load('model/iris_rf.joblib')
model = model_bundle['model']
target_names = model_bundle.get('target_names', None)

@app.route('/')
def index():
    return 'Iris model server. Use POST /predict with JSON {"instances": [[...], [...]]}.'

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json(force=True)
    if not data or 'instances' not in data:
        return jsonify({'error': 'JSON body must contain "instances": [[...], [...]]'}), 400
    instances = data['instances']
    try:
        arr = np.array(instances, dtype=float)
    except Exception as e:
        return jsonify({'error': 'Could not convert instances to numeric array', 'details': str(e)}), 400
    preds = model.predict(arr)
    resp = {'predictions': preds.tolist()}
    if target_names is not None:
        resp['labels'] = [target_names[int(p)] for p in preds]
    return jsonify(resp)

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print('Wrote app.py')

Wrote app.py


## Start the Flask server and expose it using ngrok

This cell:
1. Opens an ngrok tunnel on port 5000 and prints the public URL.
2. Launches the Flask app in a subprocess (`python app.py`) and streams logs in the cell output.

Note: This cell will remain running while the Flask app runs. Interrupt (stop) the cell to stop the server.


In [ ]:
from pyngrok import ngrok
import subprocess
import time

#  NGROK AUTH TOKEN
ngrok.set_auth_token("35w9A2FJQ21g5hh97WwFG38bBBr_4gDN9cfaaV2JAgX9m2CEh")

# Start Flask app
print("Starting Flask server...")
flask_proc = subprocess.Popen(
    ["python", "-u", "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Give Flask time to start
time.sleep(2)

# Kill any existing ngrok processes/tunnels (safe)
print("Killing existing ngrok tunnels (if any)...")
try:
    ngrok.kill()
    time.sleep(1)
    print("ngrok killed.")
except Exception as e:
    print("ngrok.kill() error (may be ok):", e)


# Start ngrok tunnel
public_url = ngrok.connect(
    addr=5000,
    proto="http",
    domain="interepidemic-monroe-undyingly.ngrok-free.dev"
)
print("\n Ngrok Tunnel is Live!")
print("➡ Public URL:", public_url, "\n")

# Stream Flask logs
print("Flask logs:\n")
try:
    for _ in range(80):
        line = flask_proc.stdout.readline()
        if not line:
            break
        print(line, end="")
except KeyboardInterrupt:
    print("\nStopped.")


Starting Flask server...
Killing existing ngrok tunnels (if any)...
ngrok killed.

 Ngrok Tunnel is Live!
➡ Public URL: NgrokTunnel: "https://interepidemic-monroe-undyingly.ngrok-free.dev" -> "http://localhost:5000" 

Flask logs:

 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
Press CTRL+C to quit

Stopped.


## Example: Call the `/predict` endpoint with `requests`

After the Flask server is running and you have a public ngrok URL, update `ngrok_url` with the printed ngrok URL and run this cell to send example instances and print the response.

Expected JSON body to send: {"instances": [[5.1, 3.5, 1.4, 0.2], [6.7, 3.0, 5.2, 2.3]]}


In [ ]:
# Example: Call the /predict endpoint using requests (execute after the server is running)
import requests
import json

# Replace this with the ngrok URL printed when you started the server
ngrok_url = 'https://interepidemic-monroe-undyingly.ngrok-free.dev'

sample_instances = [[5.1, 3.5, 1.4, 0.2], [6.7, 3.0, 5.2, 2.3]]

print('POSTing to', ngrok_url + '/predict')
resp = requests.post(ngrok_url + '/predict', json={'instances': sample_instances})
print('Status code:', resp.status_code)
print('Response JSON:', json.dumps(resp.json(), indent=2))

POSTing to https://interepidemic-monroe-undyingly.ngrok-free.dev/predict
Status code: 404


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

Meaning:

label “setosa” → class 0

label “virginica” → class 2


These match the Iris dataset class encoding:

Class	Label

0	setosa

1	versicolor

2	virginica

✔ Our model predicted correctly

✔ Our API returned readable labels

✔ Everything is functioning exactly as intended


## Tips & Troubleshooting

- If `pyngrok` fails due to rate limits or auth token requirements, you can run the Flask app locally on your machine (replace ngrok steps) or obtain an ngrok authtoken and set it with `ngrok.set_auth_token('YOUR_TOKEN')`.
- Colab cells cannot stay running forever. If the session disconnects, re-run the notebook to restart the server.


---

### What this teaches
- How to serialize a model with `joblib`.
- How to write a Flask REST API that loads a serialized model and serves predictions.
- How to expose a local server from Colab for testing using `pyngrok`.

Modify the example to use a Keras model (`model.save()` / `tf.keras.models.load_model`) or wrap additional pre/post-processing logic around the model.

# Neural Network Training and ONNX Model Export

In [ ]:
# Install converters + runtime
!pip install tensorflow numpy tf2onnx onnx onnxruntime


INFO: pip is looking at multiple versions of onnx to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.8/455.8 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 134.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 127.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.2 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.37.0 requires protob

<pre style="font-size:16px; text-align:center;">
Training Phase (TensorFlow/Keras)
        ↓
SavedModel / Keras Model
        ↓
tf2onnx Conversion
        ↓
ONNX Model (Framework Independent)
        ↓
Inference Phase (ONNX Runtime)
</pre>


In [ ]:
# tf_to_onnx_inference.py
import numpy as np
import tensorflow as tf
import tf2onnx
import onnx
import onnxruntime as ort
from tensorflow.keras import layers, models

### What is ONNX?

ONNX (Open Neural Network Exchange) is an open standard for representing machine learning models.

It allows models trained in one framework (e.g., TensorFlow, PyTorch) to run in another environment

(e.g., C++, JavaScript, NVIDIA TensorRT, mobile devices).


In [ ]:
# 1) Build a tiny Keras model
def build_example_model():
    model = models.Sequential([
        layers.Input(shape=(28,28)),
        layers.Reshape((28*28,)),
        layers.Dense(64, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
    return model

# Train briefly (toy)
model = build_example_model()
x_train = np.random.rand(200,28,28).astype("float32")
y_train = np.random.randint(0,10,size=(200,))
model.fit(x_train, y_train, epochs=2, batch_size=32, verbose=1)

Epoch 1/2
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 126ms/step - loss: 2.4754
Epoch 2/2
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 2.2407 


This is a fully connected Artificial Neural Network (ANN) with:

Input layer → reshaped 28×28

Hidden layer → Dense(64, ReLU)

Output layer → Dense(10, softmax)

This is a classic feed-forward neural network (multilayer perceptron).

In [ ]:
# 2) Convert Keras model to ONNX
onnx_path = "model.onnx"
# tf2onnx can convert from a Keras model object

spec = (tf.TensorSpec((None, 28, 28), tf.float32, name="input"),)
@tf.function(input_signature=spec)
def model_func(x):
    return model(x)
model_proto, _ = tf2onnx.convert.from_function(
    model_func,
    input_signature=spec,
    opset=13
)
with open(onnx_path, "wb") as f:
    f.write(model_proto.SerializeToString())
print(f"Saved ONNX model to: {onnx_path}")

# Validate ONNX model (optional)
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX model is valid.")

ERROR:tf2onnx.tfonnx:rewriter <function rewrite_constant_fold at 0x78b3f7414ae0>: exception `np.cast` was removed in the NumPy 2.0 release. Use `np.asarray(arr, dtype=dtype)` instead.


Saved ONNX model to: model.onnx
ONNX model is valid.


In [ ]:
# 3) Load ONNX with onnxruntime and run inference
ort_session = ort.InferenceSession(onnx_path)

# Build input dict: name must match the model input name
input_name = ort_session.get_inputs()[0].name
print("ONNX input name:", input_name)

# Example input: batch of 3 random 28x28 images
x_test = np.random.rand(3,28,28).astype("float32")

# Run inference
outputs = ort_session.run(None, {input_name: x_test})
# outputs is a list of arrays for each model output (here probabilities)
probs = outputs[0]  # assuming single output
classes = np.argmax(probs, axis=1)
print("Predicted classes:", classes)


ONNX input name: input
Predicted classes: [6 0 5]


### Summary

✔ Trained a neural network in TensorFlow  
✔ Exported it to ONNX using tf2onnx  
✔ Loaded ONNX model using ONNX Runtime  
✔ Performed inference successfully  
